# Shape-guided analogue generation for bicalutamide

Generate analogues of bicalutamide using single-step fragment operations
(ring replacement, substituent replacement, linker replacement, scaffold decoration),
rank all products by a combined 3D shape + 2D diversity score, and overlay
the best hit from each strategy on the reference conformer.

No iterative optimisation — each operation is run once, all products are scored,
and the results are ranked.


In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import (
    AllChem, Draw, rdDistGeom, rdShapeAlign, DataStructs,
    rdFingerprintGenerator,
)
from rdkit.Chem import AddHs, RemoveHs
from rdkit.Chem.Draw import MolsToGridImage
from rdkit.Chem import MolToMolBlock
from IPython.display import display

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from lacan import lacan
from lacan.replace import replace_ring, replace_substituent, replace_linker, decorate_scaffold, load_corpus

RDLogger.DisableLog('rdApp.*')
print('Imports OK')


## 1. Load reference molecule

Read the bicalutamide SDF (a pre-optimised 3D conformer) and construct the 2D mol
and ECFP4 fingerprint used for Tanimoto scoring.


In [ ]:
SDF_PATH = os.path.join('data/bicalutamide.sdf')

# 3D reference conformer (with Hs) — used for shape alignment
ref_3d  = Chem.SDMolSupplier(SDF_PATH, removeHs=False)[0]
assert ref_3d is not None, f'Could not read {SDF_PATH}'

# 2D mol — used for fragment operations and Tanimoto scoring
BICALUTAMIDE_SMILES = 'O=C(Nc1cc(c(C#N)cc1)C(F)(F)F)C(O)(C)CS(=O)(=O)c2ccc(F)cc2'
ref_mol = Chem.MolFromSmiles(BICALUTAMIDE_SMILES)

# ECFP4 fingerprint of the reference
fpgen  = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
ref_fp = fpgen.GetFingerprint(ref_mol)

print('Reference:', Chem.MolToSmiles(ref_mol))
print('Heavy atoms:', ref_mol.GetNumHeavyAtoms())
display(Draw.MolToImage(ref_mol, size=(400, 250)))


## 2. Scoring function

The combined score balances 3D shape similarity against 2D graph diversity:

```
score = shape × (1 − α × TS)
```

- `shape` — best `rdShapeAlign` score across 20 conformers (colour + shape average, 0–1)
- `TS` — ECFP4 Tanimoto similarity to the reference (0–1)
- `α` — diversity penalty weight (default 0.5; raise toward 1.0 for stronger scaffold-hopping pressure)

At α=0.5, a perfect shape match at TS=0.0 scores the full shape value,
while at TS=0.6 it scores only 70% of the shape value.


In [ ]:
# ── Tunable parameter ────────────────────────────────────────────────────────
ALPHA = 0.5   # diversity penalty weight; raise toward 1.0 for more scaffold-hopping


def ecfp4_tanimoto(mol, ref_fp=ref_fp):
    """ECFP4 Tanimoto similarity between mol and the reference."""
    fp = fpgen.GetFingerprint(mol)
    return DataStructs.TanimotoSimilarity(ref_fp, fp)


def get_alignment_scores(mols, ref=ref_3d):
    """Shape-align each mol to ref using multiple conformers.

    Embeds numConfs=20 conformers per molecule, aligns each with
    rdShapeAlign.AlignMol, and returns the best average of
    (colour_score + shape_score) / 2 across conformers.

    Parameters
    ----------
    mols : list of RDKit Mol (2D)
    ref  : RDKit Mol with a 3D conformer — use ref_3d (SDF + AddHs)

    Returns
    -------
    list of float  (raw shape scores, before diversity penalty)
    """
    scores = []
    for m in mols:
        try:
            m = AddHs(m)
            rdDistGeom.EmbedMultipleConfs(m, numConfs=20, randomSeed=0xf00d, numThreads=-1)
            conf_scores = [
                sum(rdShapeAlign.AlignMol(ref, m, probeConfId=cid)) / 2
                for cid in range(m.GetNumConformers())
            ]
            scores.append(max(conf_scores) if conf_scores else 0.0)
        except Exception:
            scores.append(0.0)
    return scores


def score_batch(mols, alpha=ALPHA):
    """Combined shape + diversity score for a list of molecules.

    score = shape * (1 - alpha * TS)
    """
    shape_scores = get_alignment_scores(mols)
    return [
        shape * (1 - alpha * ecfp4_tanimoto(m))
        for shape, m in zip(shape_scores, mols)
    ]


def shape_score(mol):
    """Combined score for a single molecule."""
    return score_batch([mol])[0]


# ── Sanity checks ─────────────────────────────────────────────────────────────
ref_shape    = get_alignment_scores([ref_mol])[0]
ref_ts       = ecfp4_tanimoto(ref_mol)   # 1.0 — same molecule
ref_combined = shape_score(ref_mol)
print(f'Reference shape score    : {ref_shape:.4f}')
print(f'Reference Tanimoto (self): {ref_ts:.4f}')
print(f'Reference combined score : {ref_combined:.4f}  (= {ref_shape:.3f} * (1 - {ALPHA}*{ref_ts:.1f}))')
print(f'\nWith alpha={ALPHA}, a perfect shape match scores:')
for ts in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    print(f'  TS={ts:.1f} → combined={ref_shape * (1 - ALPHA * ts):.4f}')


## 3. Load LACAN profile and fragment corpus


In [ ]:
profile = lacan.load_profile('chembl')
corpus  = load_corpus(min_count=200)
print(f'Profile loaded  |  corpus size: {len(corpus):,} fragments')


## 4. Generation parameters

`N_REPLACEMENTS` controls how many product molecules each operation attempts to generate.
All valid products are scored and ranked — the top `N_SHOW` are displayed per operation.


In [ ]:
N_REPLACEMENTS = 500   # attempts per operation
N_SHOW         = 9     # top-N to display per operation in the grid
SCORE_THRESHOLD = 0.05  # LACAN pre-filter (0.0 = accept all chemically valid products)


## 5. Generate analogues

Run all four operations once on the reference molecule, collect all unique products,
and score them with the combined shape + diversity metric.

This may take a few minutes — shape alignment with 20 conformers per molecule is
the bottleneck.


In [ ]:
import random
random.seed(42)

OPS = {
    'ring':   lambda m: replace_ring(
        m, profile, SCORE_THRESHOLD, fragcorpus=corpus, n_replacements=N_REPLACEMENTS),
    'sub':    lambda m: replace_substituent(
        m, profile, SCORE_THRESHOLD, fragcorpus=corpus, n_replacements=N_REPLACEMENTS),
    'linker': lambda m: replace_linker(
        m, profile, SCORE_THRESHOLD, fragcorpus=corpus, n_replacements=N_REPLACEMENTS),
}

OP_LABELS = {
    'ring':   'Ring replacement',
    'sub':    'Substituent replacement',
    'linker': 'Linker replacement',
}

# ── Run all operations ────────────────────────────────────────────────────────
raw_products = {}   # op -> list of RDKit Mol
for op_name, op_fn in OPS.items():
    prods = op_fn(ref_mol)
    # Keep only connected, sanitisable molecules
    valid = [m for m in prods if m and '.' not in Chem.MolToSmiles(m)]
    raw_products[op_name] = valid
    print(f'{OP_LABELS[op_name]:30s}: {len(valid):3d} products')

# ── Score all products ────────────────────────────────────────────────────────
print('\nScoring products (shape alignment)...')
results = {}   # op -> list of (smi, shape, ts, combined) sorted by combined desc

for op_name, mols in raw_products.items():
    if not mols:
        results[op_name] = []
        continue
    shape_scores = get_alignment_scores(mols)
    rows = []
    seen_smi = set()
    for mol, sh in zip(mols, shape_scores):
        smi = Chem.MolToSmiles(mol)
        if smi in seen_smi:
            continue
        seen_smi.add(smi)
        ts   = ecfp4_tanimoto(mol)
        comb = sh * (1 - ALPHA * ts)
        rows.append((smi, sh, ts, comb))
    rows.sort(key=lambda x: x[3], reverse=True)
    results[op_name] = rows
    print(f'  {OP_LABELS[op_name]:30s}: {len(rows):3d} unique  '
        f'best={rows[0][3]:.3f}  (shape={rows[0][1]:.3f}, TS={rows[0][2]:.3f})')


## 6. Results — top analogues per operation

Each grid shows the top-scored products for one operation.
Legend: `comb` = combined score, `sh` = raw shape score, `TS` = Tanimoto similarity.


In [ ]:
for op_name, rows in results.items():
    if not rows:
        print(f'{OP_LABELS[op_name]}: no products')
        continue
    top = rows[:N_SHOW]
    mols   = [Chem.MolFromSmiles(smi) for smi, *_ in top]
    labels = [f'comb={c:.3f}  sh={sh:.3f}  TS={ts:.2f}'
              for _, sh, ts, c in top]
    print(f'\n{OP_LABELS[op_name]} — top {len(top)}')
    display(MolsToGridImage(
        mols, molsPerRow=3, subImgSize=(340, 260), legends=labels
    ))


## 7. Combined ranking across all operations

Pool all unique products from all four operations and show the global top-N.


In [ ]:
all_rows = []   # (smi, shape, ts, combined, op)
seen_all = set()
for op_name, rows in results.items():
    for smi, sh, ts, comb in rows:
        if smi not in seen_all:
            seen_all.add(smi)
            all_rows.append((smi, sh, ts, comb, op_name))

all_rows.sort(key=lambda x: x[3], reverse=True)
print(f'Total unique products across all operations: {len(all_rows)}')
print(f'Reference combined score: {ref_combined:.4f}\n')

top_all = all_rows[:N_SHOW]
mols_all   = [Chem.MolFromSmiles(smi) for smi, *_ in top_all]
labels_all = [f'[{op}] comb={c:.3f}  TS={ts:.2f}'
              for _, sh, ts, c, op in top_all]
display(MolsToGridImage(
    mols_all, molsPerRow=3, subImgSize=(340, 260), legends=labels_all
))


## 8. 3D overlay — best hit per operation

Align the top-scoring product from each operation to the reference conformer
and visualise them together with `py3Dmol`.

- **White sticks** — bicalutamide reference
- **Blue** — best ring replacement
- **Green** — best substituent replacement
- **Orange** — best linker replacement

In [ ]:
import py3Dmol

OP_COLOURS = {
    'ring':   'blueCarbon',   # blue
    'sub':    'greenCarbon',   # green
    'linker': 'orangeCarbon',   # orange
}


def best_aligned_molblock(mol_2d, ref=ref_3d, n_confs=20, seed=0xf00d):
    """Embed mol_2d, align all confs to ref, return molblock of the best conf."""
    m = AddHs(mol_2d)
    rdDistGeom.EmbedMultipleConfs(m, numConfs=n_confs, randomSeed=seed, numThreads=-1)
    if m.GetNumConformers() == 0:
        return None, 0.0
    best_sc, best_cid = -1, 0
    for cid in range(m.GetNumConformers()):
        sc = sum(rdShapeAlign.AlignMol(ref, m, probeConfId=cid)) / 2
        if sc > best_sc:
            best_sc, best_cid = sc, cid
    rw = Chem.RWMol(m)
    for c in list(rw.GetConformers()):
        if c.GetId() != best_cid:
            rw.RemoveConformer(c.GetId())
    return MolToMolBlock(rw), best_sc


# ── Best hit per operation ────────────────────────────────────────────────────
overlay_entries = []   # (smi, op)
for op_name, rows in results.items():
    if rows:
        overlay_entries.append((rows[0][0], op_name))

print('Aligning best hits to reference...')
for smi, op in overlay_entries:
    m  = Chem.MolFromSmiles(smi)
    ts = ecfp4_tanimoto(m)
    sh = results[op][0][1]
    c  = results[op][0][3]
    print(f'  [{OP_LABELS[op]:30s}]  comb={c:.3f}  shape={sh:.3f}  TS={ts:.3f}')
    print(f'    {smi}')

# ── Build py3Dmol viewer ──────────────────────────────────────────────────────
view = py3Dmol.view(width=720, height=500)

# Reference — white sticks
view.addModel(MolToMolBlock(ref_3d), 'mol')
view.setStyle({'model': 0}, {'stick': {'colorscheme': 'whiteCarbon', 'radius': 0.12}})

# Best analogue per operation — colour-coded
for i, (smi, op) in enumerate(overlay_entries):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    molblock, sc = best_aligned_molblock(mol)
    if molblock is None:
        continue
    view.addModel(molblock, 'mol')
    view.setStyle({'model': i + 1}, {'stick': {'colorscheme': OP_COLOURS[op], 'radius': 0.12}})

view.zoomTo()
view.show()


## 9. Export results to CSV

Save the full ranked list (all operations combined) for downstream use.


In [ ]:
import csv

out_path = 'bicalutamide_analogues.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['smiles', 'operation', 'shape_score', 'tanimoto', 'combined_score'])
    for smi, sh, ts, comb, op in all_rows:
        writer.writerow([smi, OP_LABELS[op], f'{sh:.4f}', f'{ts:.4f}', f'{comb:.4f}'])

print(f'Wrote {len(all_rows)} rows to {out_path}')
